[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_06/SFM_ch6_out_of_sample/SFM_ch6_out_of_sample.ipynb)

# SFM_ch6_out_of_sample

Out-of-sample validation: the seven models fitted on 2000-2019 (Bitcoin 2014-2019) and evaluated on 2020-2026 by the mean log score, the VaR 1% exceedances (Kupiec test) and the pinball loss, with historical simulation; overfitting with Normal mixtures of 1 to 6 components (S&P 500, estimation 2017-2018); a rolling VaR 1% backtest (1000-day window, refitted every 20 days) of the Normal, Student-t and skewed-t models and historical simulation for the S&P 500 and the BET.

*Statistics of Financial Markets (SFM), Chapter 6: model selection and risk management. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_6'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
from scipy import optimize

from scipy.special import gammaln

START = '2000-01-01'

STOCK_START = '2010-01-01'

ASSETS = ['bet', 'sp500', 'dax', 'btc']

STOCKS = ['tlv', 'snp', 'brd']

BAD_DAYS = {'tlv': ['2016-05-30', '2016-05-31']}

MODELS = ['Normal', 'Student-t', 'Skewed-t', 'GED', 'Mixture', 'NIG', 'Stable']

K = {'Normal': 2, 'Student-t': 3, 'Skewed-t': 4, 'GED': 3, 'Mixture': 5, 'NIG': 4, 'Stable': 4}

MODEL_COL = {'Normal': '#CD0000', 'Student-t': '#2E7D32', 'Skewed-t': '#8E44AD', 'GED': '#E67E22', 'Mixture': '#17A2B8', 'NIG': '#B5853F', 'Stable': '#1A3A6E', 'Data': 'black', 'Historical': '#DC3545'}

SHORT = {'bet': 'BET', 'sp500': 'S&P 500', 'dax': 'DAX', 'btc': 'Bitcoin', 'tlv': 'TLV', 'snp': 'SNP', 'brd': 'BRD'}

COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F', 'tlv': '#E67E22', 'snp': '#2E7D32', 'brd': '#8E44AD'}

SEED = 2026

SPLIT = '2019-12-31'

ALPHA_VAR = 0.01

PANEL = ['Normal', 'Student-t', 'Skewed-t', 'Stable']

ALPHA_ES = 0.025

BLOCK = 20

def stable_cf(t, alpha, beta, gamma=1.0, delta=0.0, param=0):
    """Characteristic function E exp(itX) of S(alpha, beta, gamma, delta; param), Nolan's S0 (param=0) or S1."""
    t = np.asarray(t, float)
    a = np.abs(gamma * t)
    if alpha != 1:
        tan = np.tan(np.pi * alpha / 2)
        if param == 0:      # -|gt|^a [1 + i b tan(pi a/2) sign(t) (|gt|^(1-a) - 1)], written without 0^(1-a)
            psi = -a ** alpha - 1j * beta * tan * (gamma * t - np.sign(t) * a ** alpha)
        else:               # -|gt|^a [1 - i b tan(pi a/2) sign(t)]
            psi = -a ** alpha * (1 - 1j * beta * tan * np.sign(t))
    else:
        arg = a if param == 0 else np.abs(t)
        lg = np.log(np.where(arg > 0, arg, 1.0))
        psi = -a * (1 + 1j * beta * (2 / np.pi) * np.sign(t) * lg)
    return np.exp(psi + 1j * delta * t)


def s0_to_s1(alpha, beta, gamma, delta0):
    """Location in S1 from the location in S0."""
    if alpha != 1:
        return delta0 - beta * gamma * np.tan(np.pi * alpha / 2)
    return delta0 - beta * (2 / np.pi) * gamma * np.log(gamma)


class StableGrid:
    """Density and distribution function of the standard S0(alpha, beta, 1, 0) law on a fine grid, by FFT
    inversion of the characteristic function, f(x) = (1/2pi) int phi(t) exp(-itx) dt (Mittnik et al., 1999).
    Grid step h = 0.01 on [-327.68, 327.67]; the power-law tail approximation is used beyond the grid."""

    def __init__(self, alpha, beta, h=0.01, N=2 ** 16):
        k = np.fft.fftfreq(N, d=1.0 / N)                        # 0, 1, ..., N/2-1, -N/2, ..., -1
        dt = 2 * np.pi / (N * h)
        phi = stable_cf(k * dt, alpha, beta, 1.0, 0.0, 0) * np.where(k % 2 == 0, 1.0, -1.0)
        f = np.real(np.fft.fft(phi)) * dt / (2 * np.pi)
        self.alpha, self.beta, self.h = alpha, beta, h
        self.x = (np.arange(N) - N / 2) * h
        self.f = np.maximum(f, 1e-300)
        self.logf = np.log(self.f)
        # tail constant: P(X > x) ~ c_alpha (1 + beta) x^(-alpha), c_alpha = Gamma(alpha) sin(pi alpha / 2) / pi
        from scipy.special import gamma as G
        self.c = 0.0 if alpha >= 2 else G(alpha) * np.sin(np.pi * alpha / 2) / np.pi
        left = self.c * (1 - beta) * abs(self.x[0]) ** (-alpha) if alpha < 2 else 0.0
        right = self.c * (1 + beta) * self.x[-1] ** (-alpha) if alpha < 2 else 0.0
        cells = 0.5 * (self.f[1:] + self.f[:-1]) * h
        self.F = np.clip(left + np.concatenate([[0.0], np.cumsum(cells)]), 0.0, 1.0)           # P(X <= x), from the left
        self.S = np.clip(right + np.concatenate([np.cumsum(cells[::-1])[::-1], [0.0]]), 0.0, 1.0)  # P(X > x), from the right

    def pdf(self, z):
        z = np.asarray(z, float)
        out = np.exp(np.interp(z, self.x, self.logf))
        far = np.abs(z) > self.x[-1]
        if far.any() and self.alpha < 2:
            out[far] = self.alpha * self.c * (1 + np.sign(z[far]) * self.beta) * np.abs(z[far]) ** (-self.alpha - 1)
        return out

    def cdf(self, z):
        z = np.asarray(z, float)
        return np.where(z <= 0, np.interp(z, self.x, self.F), 1 - np.interp(z, self.x, self.S))

    def sf(self, z):
        return 1 - self.cdf(z) if np.ndim(z) == 0 else np.where(np.asarray(z) > 0, np.interp(z, self.x, self.S), 1 - self.cdf(z))

    def ppf(self, p):
        i = np.searchsorted(self.F, 1e-12) if self.F[0] < 1e-12 else 0
        return np.interp(p, self.F[i:], self.x[i:])


def mcculloch(x):
    """McCulloch (1986) quantile estimator of (alpha, beta, gamma, delta0, delta1) from a sample;
    sample quantiles: x_(i) is the (2i - 1)/(2n) quantile, linear interpolation."""
    q05, q25, q50, q75, q95 = np.percentile(np.asarray(x, float), [5, 25, 50, 75, 95], method='hazen')
    return mcculloch_quantiles(q05, q25, q50, q75, q95)


def mcculloch_quantiles(q05, q25, q50, q75, q95):
    """McCulloch (1986) estimates from the 5%, 25%, 50%, 75% and 95% quantiles.
    nu_alpha = (q95 - q05)/(q75 - q25), nu_beta = (q95 + q05 - 2 q50)/(q95 - q05); alpha and beta from
    Tables III-IV, gamma = (q75 - q25)/phi_3(alpha, beta) (Table V), the S0 location from Table VII."""
    from scipy.interpolate import RegularGridInterpolator as RGI
    nu_a_grid = [2.439, 2.5, 2.6, 2.7, 2.8, 3.0, 3.2, 3.5, 4.0, 5.0, 6.0, 8.0, 10.0, 15.0, 25.0]
    nu_b_grid = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
    T3 = [[2.000, 2.000, 2.000, 2.000, 2.000, 2.000, 2.000], [1.916, 1.924, 1.924, 1.924, 1.924, 1.924, 1.924],
          [1.808, 1.813, 1.829, 1.829, 1.829, 1.829, 1.829], [1.729, 1.730, 1.737, 1.745, 1.745, 1.745, 1.745],
          [1.664, 1.663, 1.663, 1.668, 1.676, 1.676, 1.676], [1.563, 1.560, 1.553, 1.548, 1.547, 1.547, 1.547],
          [1.484, 1.480, 1.471, 1.460, 1.448, 1.438, 1.438], [1.391, 1.386, 1.378, 1.364, 1.337, 1.318, 1.318],
          [1.279, 1.273, 1.266, 1.250, 1.210, 1.184, 1.150], [1.128, 1.121, 1.114, 1.101, 1.067, 1.027, 0.973],
          [1.029, 1.021, 1.014, 1.004, 0.974, 0.935, 0.874], [0.896, 0.892, 0.884, 0.883, 0.855, 0.823, 0.769],
          [0.818, 0.812, 0.806, 0.801, 0.780, 0.756, 0.691], [0.698, 0.695, 0.692, 0.689, 0.676, 0.656, 0.597],
          [0.593, 0.590, 0.588, 0.586, 0.579, 0.563, 0.513]]
    T4 = [[0, 2.160, 1.000, 1.000, 1.000, 1.000, 1.000], [0, 1.592, 3.390, 1.000, 1.000, 1.000, 1.000],
          [0, 0.759, 1.800, 1.000, 1.000, 1.000, 1.000], [0, 0.482, 1.048, 1.694, 1.000, 1.000, 1.000],
          [0, 0.360, 0.760, 1.232, 2.229, 1.000, 1.000], [0, 0.253, 0.518, 0.823, 1.575, 1.000, 1.000],
          [0, 0.203, 0.410, 0.632, 1.244, 1.906, 1.000], [0, 0.165, 0.332, 0.499, 0.943, 1.560, 1.000],
          [0, 0.136, 0.271, 0.404, 0.689, 1.230, 2.195], [0, 0.109, 0.216, 0.323, 0.539, 0.827, 1.917],
          [0, 0.096, 0.190, 0.284, 0.472, 0.693, 1.759], [0, 0.082, 0.163, 0.243, 0.412, 0.601, 1.596],
          [0, 0.074, 0.147, 0.220, 0.377, 0.546, 1.482], [0, 0.064, 0.128, 0.191, 0.330, 0.478, 1.362],
          [0, 0.056, 0.112, 0.167, 0.285, 0.428, 1.274]]
    a_grid = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
    b_grid = [0.0, 0.25, 0.5, 0.75, 1.0]
    T5 = [[2.588, 3.073, 4.534, 6.636, 9.144], [2.337, 2.634, 3.542, 4.808, 6.247], [2.189, 2.392, 3.004, 3.844, 4.775],
          [2.098, 2.244, 2.676, 3.265, 3.912], [2.040, 2.149, 2.461, 2.886, 3.356], [2.000, 2.085, 2.311, 2.624, 2.973],
          [1.980, 2.040, 2.205, 2.435, 2.696], [1.965, 2.007, 2.125, 2.294, 2.491], [1.955, 1.984, 2.067, 2.188, 2.333],
          [1.946, 1.967, 2.022, 2.106, 2.211], [1.939, 1.952, 1.988, 2.045, 2.116], [1.933, 1.940, 1.962, 1.997, 2.043],
          [1.927, 1.930, 1.943, 1.961, 1.987], [1.921, 1.922, 1.927, 1.936, 1.947], [1.914, 1.915, 1.916, 1.918, 1.921],
          [1.908, 1.908, 1.908, 1.908, 1.908]]
    T7 = [[0, -0.061, -0.279, -0.659, -1.198], [0, -0.078, -0.272, -0.581, -0.997], [0, -0.089, -0.262, -0.520, -0.853],
          [0, -0.096, -0.250, -0.469, -0.742], [0, -0.099, -0.237, -0.424, -0.652], [0, -0.098, -0.223, -0.380, -0.576],
          [0, -0.095, -0.208, -0.346, -0.508], [0, -0.090, -0.192, -0.310, -0.447], [0, -0.084, -0.173, -0.276, -0.390],
          [0, -0.075, -0.154, -0.241, -0.335], [0, -0.066, -0.134, -0.206, -0.283], [0, -0.056, -0.111, -0.170, -0.232],
          [0, -0.043, -0.088, -0.132, -0.179], [0, -0.030, -0.061, -0.092, -0.123], [0, -0.017, -0.032, -0.049, -0.064],
          [0, 0.000, 0.000, 0.000, 0.000]]
    psi1 = RGI((nu_a_grid, nu_b_grid), np.array(T3), bounds_error=False, fill_value=None)
    psi2 = RGI((nu_a_grid, nu_b_grid), np.array(T4), bounds_error=False, fill_value=None)
    phi3 = RGI((a_grid, b_grid), np.array(T5), bounds_error=False, fill_value=None)
    phi5 = RGI((a_grid, b_grid), np.array(T7), bounds_error=False, fill_value=None)
    nu_a = (q95 - q05) / (q75 - q25)
    nu_b = (q95 + q05 - 2 * q50) / (q95 - q05)
    if nu_a >= 2.439:
        alpha = float(np.clip(psi1([[nu_a, abs(nu_b)]])[0], 0.5, 2.0))
        beta = float(np.clip(np.sign(nu_b) * psi2([[nu_a, abs(nu_b)]])[0], -1.0, 1.0))
    else:
        alpha, beta = 2.0, 0.0
    gamma = float((q75 - q25) / phi3([[alpha, abs(beta)]])[0])
    delta0 = float(q50 + gamma * np.sign(beta) * phi5([[alpha, abs(beta)]])[0])   # zeta in McCulloch (1986)
    delta1 = float(s0_to_s1(alpha, beta, gamma, delta0)) if alpha != 1 else delta0
    return {'alpha': alpha, 'beta': beta, 'gamma': gamma, 'delta0': delta0, 'delta1': delta1,
            'nu_alpha': float(nu_a), 'nu_beta': float(nu_b),
            'q05': float(q05), 'q25': float(q25), 'q50': float(q50), 'q75': float(q75), 'q95': float(q95)}


def stable_nll(p, x):
    """Negative log-likelihood of S0(alpha, beta, gamma, delta0) at p = (alpha, beta, log gamma, delta0)."""
    a, b, lg, d = p
    if not (1.01 <= a <= 1.999 and -0.999 <= b <= 0.999):
        return 1e12
    G = StableGrid(a, b)
    return -np.sum(np.log(G.pdf((x - d) / np.exp(lg)))) + len(x) * lg


def stable_mle(x, start=None, se=True):
    """Maximum likelihood estimate of (alpha, beta, gamma, delta0) in S0 (Nelder-Mead from the McCulloch estimate),
    with standard errors from the numerical Hessian of the log-likelihood; delta1 is the S1 location."""
    from scipy.optimize import minimize
    x = np.asarray(x, float)
    m = start or mcculloch(x)
    p0 = [min(max(m['alpha'], 1.05), 1.98), float(np.clip(m['beta'], -0.95, 0.95)), np.log(m['gamma']), m['delta0']]
    res = minimize(stable_nll, p0, args=(x,), method='Nelder-Mead',
                   options={'xatol': 1e-5, 'fatol': 1e-4, 'maxiter': 4000, 'maxfev': 8000})
    a, b, lg, d = res.x
    out = {'alpha': float(a), 'beta': float(b), 'gamma': float(np.exp(lg)), 'delta0': float(d),
           'delta1': float(s0_to_s1(a, b, np.exp(lg), d)), 'loglik': float(-res.fun), 'n': len(x)}
    if se:
        th = np.array([a, b, np.exp(lg), d])

        def f(v):
            return stable_nll([v[0], v[1], np.log(v[2]), v[3]], x)
        steps = np.array([1e-3, 1e-2, 1e-3 * th[2], 1e-3 * th[2]])
        H = np.zeros((4, 4))
        for i in range(4):
            for j in range(i, 4):
                ei, ej = np.eye(4)[i] * steps[i], np.eye(4)[j] * steps[j]
                H[i, j] = H[j, i] = (f(th + ei + ej) - f(th + ei - ej) - f(th - ei + ej) + f(th - ei - ej)) / (4 * steps[i] * steps[j])
        try:
            cov = np.linalg.inv(H)
            sd = np.sqrt(np.clip(np.diag(cov), 0, None))
        except np.linalg.LinAlgError:
            sd = np.full(4, np.nan)
        out.update({'se_alpha': float(sd[0]), 'se_beta': float(sd[1]), 'se_gamma': float(sd[2]), 'se_delta0': float(sd[3])})
    return out


def returns(k, start=None, end=None):
    """Daily log returns in %: indices since 2000, Bitcoin over its full history, Bucharest stocks since 2010
    (the two erroneous Banca Transilvania days of May 2016 removed)."""
    s0 = start or (STOCK_START if k in STOCKS else START if k != 'btc' else None)
    r = log_returns(k, s0)
    if end:
        r = r.loc[:end]
    if k in BAD_DAYS:
        r = r.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return r


def skewt_consts(nu, lam):
    """Constants a, b, c of the skewed Student-t of Hansen (1994), standardised to mean 0 and variance 1."""
    c = np.exp(gammaln((nu + 1) / 2) - gammaln(nu / 2)) / np.sqrt(np.pi * (nu - 2))
    a = 4 * lam * c * (nu - 2) / (nu - 1)
    b = np.sqrt(1 + 3 * lam ** 2 - a ** 2)
    return a, b, c


def skewt_logpdf(x, mu, sigma, nu, lam):
    """Log-density of Hansen's skewed-t with mean mu, standard deviation sigma, degrees of freedom nu > 2 and
    skewness lam in (-1, 1); lam < 0: longer left tail; lam = 0: the Student-t with variance sigma^2."""
    a, b, c = skewt_consts(nu, lam)
    z = (np.asarray(x, float) - mu) / sigma
    s = np.where(z < -a / b, 1 - lam, 1 + lam)
    return np.log(b * c / sigma) - (nu + 1) / 2 * np.log1p(((b * z + a) / s) ** 2 / (nu - 2))


def skewt_cdf(x, mu, sigma, nu, lam):
    a, b, c = skewt_consts(nu, lam)
    z = (np.asarray(x, float) - mu) / sigma
    k = np.sqrt(nu / (nu - 2))
    lo = (1 - lam) * stats.t.cdf(k * (b * z + a) / (1 - lam), nu)
    hi = (1 - lam) / 2 + (1 + lam) * (stats.t.cdf(k * (b * z + a) / (1 + lam), nu) - 0.5)
    return np.where(z < -a / b, lo, hi)


def skewt_ppf(p, mu, sigma, nu, lam):
    a, b, c = skewt_consts(nu, lam)
    p = np.asarray(p, float)
    k = np.sqrt((nu - 2) / nu)
    lo = ((1 - lam) * k * stats.t.ppf(np.clip(p / (1 - lam), 1e-300, 1), nu) - a) / b
    hi = ((1 + lam) * k * stats.t.ppf(np.clip(0.5 + (p - (1 - lam) / 2) / (1 + lam), 0, 1 - 1e-16), nu) - a) / b
    return mu + sigma * np.where(p < (1 - lam) / 2, lo, hi)


def mixture_em(x, k=2, starts=4, iters=600, tol=1e-9, seed=SEED):
    """Maximum likelihood fit of a k-component Normal mixture by the EM algorithm (several starts, the best kept)."""
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    best = None
    for s in range(starts):
        if s == 0:      # start: equal weights, means at quantiles, spread scales
            w = np.full(k, 1 / k)
            mu = np.quantile(x, (np.arange(k) + 0.5) / k) * 0.2
            sd = x.std() * np.linspace(0.6, 2.0, k)
        else:
            w = rng.dirichlet(np.ones(k) * 3)
            mu = rng.choice(x, k) * 0.3
            sd = x.std() * rng.uniform(0.4, 2.5, k)
        ll_old = -np.inf
        for _ in range(iters):
            lp = np.log(w) + stats.norm.logpdf(x[:, None], mu, sd)
            m = lp.max(axis=1, keepdims=True)
            lse = m[:, 0] + np.log(np.exp(lp - m).sum(axis=1))
            ll = lse.sum()
            g = np.exp(lp - lse[:, None])
            nk = g.sum(axis=0) + 1e-12
            w = nk / len(x)
            mu = (g * x[:, None]).sum(axis=0) / nk
            sd = np.sqrt((g * (x[:, None] - mu) ** 2).sum(axis=0) / nk)
            sd = np.maximum(sd, 1e-3 * x.std())
            if abs(ll - ll_old) < tol * abs(ll):
                break
            ll_old = ll
        if best is None or ll > best['loglik']:
            o = np.argsort(sd)
            best = {'w': w[o].tolist(), 'mu': mu[o].tolist(), 'sd': sd[o].tolist(), 'loglik': float(ll)}
    return best


def mixture_logpdf(x, w, mu, sd):
    lp = np.log(np.asarray(w)) + stats.norm.logpdf(np.asarray(x, float)[:, None], mu, sd)
    m = lp.max(axis=1, keepdims=True)
    return m[:, 0] + np.log(np.exp(lp - m).sum(axis=1))


def mixture_cdf(x, w, mu, sd):
    return (np.asarray(w) * stats.norm.cdf(np.asarray(x, float)[:, None], mu, sd)).sum(axis=1)


def mixture_ppf(p, w, mu, sd):
    p = np.atleast_1d(np.asarray(p, float))
    lo, hi = min(mu) - 60 * max(sd), max(mu) + 60 * max(sd)
    return np.array([optimize.brentq(lambda z: mixture_cdf([z], w, mu, sd)[0] - q, lo, hi) for q in p])


def nig_fit(x):
    """Maximum likelihood fit of the NIG law (scipy.stats.norminvgauss: tail a > 0, skewness |b| < a, loc, scale)."""
    x = np.asarray(x, float)

    def nll(p):
        a = np.exp(p[0])
        b = a * np.tanh(p[1])
        v = -np.sum(stats.norminvgauss.logpdf(x, a, b, p[2], np.exp(p[3])))
        return v if np.isfinite(v) else 1e12
    sd = x.std()
    p0 = [np.log(1.0), 0.0, np.median(x), np.log(sd)]
    res = optimize.minimize(nll, p0, method='Nelder-Mead', options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 6000, 'maxfev': 12000})
    a = float(np.exp(res.x[0]))
    return {'a': a, 'b': float(a * np.tanh(res.x[1])), 'loc': float(res.x[2]), 'scale': float(np.exp(res.x[3])), 'loglik': float(-res.fun)}


def skewt_fit(x, start=None):
    """Maximum likelihood fit of Hansen's skewed-t (mu, sigma, nu, lam)."""
    x = np.asarray(x, float)

    def nll(p):
        nu, lam = 2 + np.exp(p[2]), np.tanh(p[3])
        if nu - 2 < 1e-6 or abs(lam) > 0.999:
            return 1e12
        v = -np.sum(skewt_logpdf(x, p[0], np.exp(p[1]), nu, lam))
        return v if np.isfinite(v) else 1e12
    if start is None:
        nu0 = stats.t.fit(x)[0] if len(x) < 3000 else 4.0
        start = [x.mean(), np.log(x.std()), np.log(max(nu0 - 2, 0.5)), 0.0]
    res = optimize.minimize(nll, start, method='Nelder-Mead', options={'xatol': 1e-7, 'fatol': 1e-7, 'maxiter': 8000, 'maxfev': 16000})
    return {'mu': float(res.x[0]), 'sigma': float(np.exp(res.x[1])), 'nu': float(2 + np.exp(res.x[2])),
            'lam': float(np.tanh(res.x[3])), 'loglik': float(-res.fun)}


def ged_fit(x):
    """Maximum likelihood fit of the GED (scipy.stats.gennorm: shape beta, loc, scale); beta = 2 is the Normal law,
    beta = 1 the Laplace law, beta < 2 heavier tails than the Normal law."""
    x = np.asarray(x, float)

    def nll(p):
        v = -np.sum(stats.gennorm.logpdf(x, np.exp(p[0]), p[1], np.exp(p[2])))
        return v if np.isfinite(v) else 1e12
    res = optimize.minimize(nll, [np.log(1.2), np.median(x), np.log(x.std())], method='Nelder-Mead',
                            options={'xatol': 1e-7, 'fatol': 1e-7, 'maxiter': 4000})
    return {'beta': float(np.exp(res.x[0])), 'loc': float(res.x[1]), 'scale': float(np.exp(res.x[2])), 'loglik': float(-res.fun)}


def t_fit(x):
    """Maximum likelihood fit of the location-scale Student-t (nu, loc, scale)."""
    x = np.asarray(x, float)
    nu, loc, sc = stats.t.fit(x)
    return {'nu': float(nu), 'loc': float(loc), 'scale': float(sc), 'loglik': float(np.sum(stats.t.logpdf(x, nu, loc, sc)))}


def fit_model(name, x):
    """Maximum likelihood fit of one candidate model; returns its parameters and the maximised log-likelihood."""
    x = np.asarray(x, float)
    if name == 'Normal':
        mu, sd = x.mean(), x.std(ddof=0)
        return {'mu': float(mu), 'sigma': float(sd), 'loglik': float(np.sum(stats.norm.logpdf(x, mu, sd)))}
    if name == 'Student-t':
        return t_fit(x)
    if name == 'Skewed-t':
        return skewt_fit(x)
    if name == 'GED':
        return ged_fit(x)
    if name == 'Mixture':
        return mixture_em(x, 2)
    if name == 'NIG':
        return nig_fit(x)
    if name == 'Stable':
        s = stable_mle(x, se=False)
        return dict(s, loglik=float(s['loglik']))
    raise ValueError(name)


def logpdf(name, x, p):
    x = np.asarray(x, float)
    if name == 'Normal':
        return stats.norm.logpdf(x, p['mu'], p['sigma'])
    if name == 'Student-t':
        return stats.t.logpdf(x, p['nu'], p['loc'], p['scale'])
    if name == 'Skewed-t':
        return skewt_logpdf(x, p['mu'], p['sigma'], p['nu'], p['lam'])
    if name == 'GED':
        return stats.gennorm.logpdf(x, p['beta'], p['loc'], p['scale'])
    if name == 'Mixture':
        return mixture_logpdf(x, p['w'], p['mu'], p['sd'])
    if name == 'NIG':
        return stats.norminvgauss.logpdf(x, p['a'], p['b'], p['loc'], p['scale'])
    if name == 'Stable':
        return np.log(StableGrid(p['alpha'], p['beta']).pdf((x - p['delta0']) / p['gamma']) / p['gamma'])
    raise ValueError(name)


def cdf(name, x, p):
    x = np.asarray(x, float)
    if name == 'Normal':
        return stats.norm.cdf(x, p['mu'], p['sigma'])
    if name == 'Student-t':
        return stats.t.cdf(x, p['nu'], p['loc'], p['scale'])
    if name == 'Skewed-t':
        return skewt_cdf(x, p['mu'], p['sigma'], p['nu'], p['lam'])
    if name == 'GED':
        return stats.gennorm.cdf(x, p['beta'], p['loc'], p['scale'])
    if name == 'Mixture':
        return mixture_cdf(x, p['w'], p['mu'], p['sd'])
    if name == 'NIG':
        return stats.norminvgauss.cdf(x, p['a'], p['b'], p['loc'], p['scale'])
    if name == 'Stable':
        return StableGrid(p['alpha'], p['beta']).cdf((x - p['delta0']) / p['gamma'])
    raise ValueError(name)


def ppf(name, q, p):
    q = np.asarray(q, float)
    if name == 'Normal':
        return stats.norm.ppf(q, p['mu'], p['sigma'])
    if name == 'Student-t':
        return stats.t.ppf(q, p['nu'], p['loc'], p['scale'])
    if name == 'Skewed-t':
        return skewt_ppf(q, p['mu'], p['sigma'], p['nu'], p['lam'])
    if name == 'GED':
        return stats.gennorm.ppf(q, p['beta'], p['loc'], p['scale'])
    if name == 'Mixture':
        return mixture_ppf(q, p['w'], p['mu'], p['sd'])
    if name == 'NIG':
        return stats.norminvgauss.ppf(q, p['a'], p['b'], p['loc'], p['scale'])
    if name == 'Stable':
        return p['delta0'] + p['gamma'] * StableGrid(p['alpha'], p['beta']).ppf(q)
    raise ValueError(name)


def var_es(name, p, a_var=ALPHA_VAR, a_es=ALPHA_ES, m=4000):
    """VaR_a = -q_a and ES_a = -(1/a) int_0^a q_u du (midpoint rule on a fine grid of u, refined near 0)."""
    var = -float(np.ravel(ppf(name, a_var, p))[0])
    u = a_es * np.concatenate([np.logspace(-6, -2, m // 2, endpoint=False), np.linspace(0.01, 1, m // 2 + 1)])
    mid, du = 0.5 * (u[1:] + u[:-1]), np.diff(u)
    qs = ppf(name, mid, p)
    es = -float(np.sum(qs * du) + np.ravel(ppf(name, u[0] / 2, p))[0] * u[0]) / a_es
    return var, es


def kupiec(x, n, p=ALPHA_VAR):
    """Kupiec (1995) unconditional coverage test: LR = -2 ln[(1-p)^(n-x) p^x / ((1-x/n)^(n-x) (x/n)^x)] ~ chi2(1)."""
    phat = x / n
    l0 = (n - x) * np.log(1 - p) + x * np.log(p)
    l1 = (n - x) * np.log(1 - phat) + (x * np.log(phat) if x > 0 else 0.0)
    lr = -2 * (l0 - l1)
    return {'x': int(x), 'n': int(n), 'rate': float(phat), 'lr': float(lr), 'p': float(stats.chi2.sf(lr, 1))}


def pinball(r, q, a=ALPHA_VAR):
    """Quantile (pinball) loss of a VaR forecast q (a quantile of the return): mean of (a - 1{r < q})(r - q)."""
    r = np.asarray(r, float)
    return float(np.mean((a - (r < q)) * (r - q)))


def out_of_sample(names=ASSETS, models=MODELS, split=SPLIT):
    """Fit each model until `split`; on the later days: mean log score, VaR 1% exceedances (Kupiec) and pinball loss."""
    out = {}
    for k in names:
        r = returns(k)
        est, test = r.loc[:split].values, r.loc[split:].iloc[1:].values
        row = {'n_est': len(est), 'n_test': len(test), 'emp_var1_est': float(-np.quantile(est, ALPHA_VAR))}
        for m in models:
            p = fit_model(m, est)
            q = float(np.ravel(ppf(m, ALPHA_VAR, p))[0])
            ls = logpdf(m, test, p)
            row[m] = {'logscore': float(np.mean(ls)), 'loglik_in': p['loglik'] / len(est), 'aic': 2 * K[m] - 2 * p['loglik'],
                      'var1': -q, 'kupiec': kupiec(int(np.sum(test < q)), len(test)), 'pinball': pinball(test, q)}
        qh = np.quantile(est, ALPHA_VAR)
        row['Historical'] = {'var1': float(-qh), 'kupiec': kupiec(int(np.sum(test < qh)), len(test)), 'pinball': pinball(test, qh)}
        aics = {m: row[m]['aic'] for m in models}
        row['best_aic_est'] = min(aics, key=aics.get)
        row['best_logscore'] = max(models, key=lambda m: row[m]['logscore'])
        row['best_pinball'] = min(list(models) + ['Historical'], key=lambda m: row[m]['pinball'])
        out[k] = row
        print('  oos', k, row['best_aic_est'], row['best_logscore'], row['best_pinball'])
    return out


def fig_oos(oos, names=ASSETS, models=('Normal', 'Student-t', 'Skewed-t', 'GED', 'Mixture', 'NIG', 'Stable'), save=True):
    """Left: out-of-sample mean log score minus that of the Normal model. Right: VaR 1% exceedance rate on the test days."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
    w = 0.8 / len(models)
    xs = np.arange(len(names))
    for j, m in enumerate(models):
        off = (j - (len(models) - 1) / 2) * w
        axes[0].bar(xs + off, [oos[k][m]['logscore'] - oos[k]['Normal']['logscore'] for k in names], width=w, color=MODEL_COL[m], label=m)
        axes[1].bar(xs + off, [100 * oos[k][m]['kupiec']['rate'] for k in names], width=w, color=MODEL_COL[m], label='_')
    axes[1].axhline(1, color='black', lw=0.9, ls='--', label='target 1%')
    axes[0].set_title('Out-of-sample log score minus Normal')
    axes[1].set_title('VaR 1% exceedances on the test days (%)')
    for ax in axes:
        ax.set_xticks(xs)
        ax.set_xticklabels([LABELS[k] for k in names])
    st.fig_legend_bottom(fig, ncol=8, y=0.0)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch6_oos')


def overfit_mixtures(k='sp500', est=('2017-01-01', '2018-12-31'), test_end=None, kmax=6, save=True):
    """Overfitting: Normal mixtures with 1-6 components fitted on two years of returns; in-sample log-likelihood,
    AIC, BIC and the out-of-sample mean log score on the following years."""
    r = returns(k)
    x = r.loc[est[0]:est[1]].values
    y = r.loc[est[1]:].iloc[1:].values if test_end is None else r.loc[est[1]:test_end].iloc[1:].values
    out = {}
    for c in range(1, kmax + 1):
        p = mixture_em(x, c, starts=12, seed=SEED + c)
        kk = 3 * c - 1
        out[c] = {'k': kk, 'loglik': p['loglik'], 'aic': 2 * kk - 2 * p['loglik'], 'bic': kk * np.log(len(x)) - 2 * p['loglik'],
                  'in': p['loglik'] / len(x), 'out': float(np.mean(mixture_logpdf(y, p['w'], p['mu'], p['sd'])))}
    cs = list(out)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
    axes[0].plot(cs, [out[c]['in'] for c in cs], 'o-', color='#1A3A6E', label='in sample (estimation years)')
    axes[0].plot(cs, [out[c]['out'] for c in cs], 's-', color='#CD0000', label='out of sample (later years)')
    axes[0].set_title('Mean log-likelihood per day')
    axes[1].plot(cs, [out[c]['aic'] - min(out[d]['aic'] for d in cs) for c in cs], 'o-', color='#2E7D32', label='Delta AIC')
    axes[1].plot(cs, [out[c]['bic'] - min(out[d]['bic'] for d in cs) for c in cs], 's-', color='#8E44AD', label='Delta BIC')
    axes[1].set_title('Information criteria (0 = selected)')
    for ax in axes:
        ax.set_xlabel('Number of Normal components')
        ax.set_xticks(cs)
        st.legend_outside_bottom(ax, ncol=2, y=-0.18)
    fig.tight_layout()
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch6_overfit')
    return {'n_in': len(x), 'n_out': len(y), 'est': list(est),
            'res': {str(c): v for c, v in out.items()},
            'best_in': max(cs, key=lambda c: out[c]['in']), 'best_out': max(cs, key=lambda c: out[c]['out']),
            'best_aic': min(cs, key=lambda c: out[c]['aic']), 'best_bic': min(cs, key=lambda c: out[c]['bic'])}


def rolling_var(k='sp500', window=1000, step=20, models=('Normal', 'Student-t', 'Skewed-t')):
    """Rolling VaR 1%: each model refitted every `step` days on the previous `window` days; historical simulation
    (empirical 1% quantile of the window) as a benchmark."""
    r = returns(k)
    x = r.values
    idx = r.index
    V = {m: np.full(len(x), np.nan) for m in list(models) + ['Historical']}
    for s in range(window, len(x), step):
        w = x[s - window:s]
        for m in models:
            V[m][s:s + step] = np.ravel(ppf(m, ALPHA_VAR, fit_model(m, w)))[0]
        V['Historical'][s:s + step] = np.quantile(w, ALPHA_VAR)
    df = pd.DataFrame(V, index=idx)
    df['r'] = x
    df = df.iloc[window:]
    res = {m: kupiec(int(np.sum(df['r'] < df[m])), len(df)) for m in V}
    for m in V:
        e = (df['r'] < df[m]).astype(int)
        res[m]['max_year'] = int(e.groupby(df.index.year).sum().max())
        res[m]['max_year_y'] = int(e.groupby(df.index.year).sum().idxmax())
    res['first'] = df.index[0].date().isoformat()
    return df, res


def fig_rolling_var(df, k='sp500', since='2018-01-01', save=True):
    d = df.loc[since:]
    fig, ax = plt.subplots(figsize=(10.5, 3.9))
    ax.plot(d.index, d['r'], color='#17A2B8', lw=0.6, label=f'{LABELS[k]} daily log return (%)')
    for m, c in [('Normal', MODEL_COL['Normal']), ('Student-t', MODEL_COL['Student-t']), ('Historical', '#8E44AD')]:
        ax.plot(d.index, d[m], color=c, lw=1.3, label=f'VaR 1% line, {m}' if m != 'Historical' else 'VaR 1% line, historical simulation')
    e = d['r'] < d['Student-t']
    ax.plot(d.index[e], d['r'][e], 'v', ms=5, color='black', label='exceedance of the Student-t VaR 1%')
    ax.set_ylabel('%')
    st.legend_outside_bottom(ax, ncol=3, y=-0.12)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch6_rolling_var')

## Run

In [ ]:
oos = out_of_sample()
fig_oos(oos)
print(overfit_mixtures('sp500'))
df, res = rolling_var('sp500')
print(res)
fig_rolling_var(df, 'sp500')

![sfm_ch6_oos](./sfm_ch6_oos.png)

Output: `sfm_ch6_oos.pdf`

![sfm_ch6_overfit](./sfm_ch6_overfit.png)

Output: `sfm_ch6_overfit.pdf`

![sfm_ch6_rolling_var](./sfm_ch6_rolling_var.png)

Output: `sfm_ch6_rolling_var.pdf`